# 12 · Production Patterns  🔴 Expert

1. Retries for flaky nodes (`RetryPolicy`)
2. Node caching (`CachePolicy`)
3. Model fallbacks
4. Time travel: replay & fork from any checkpoint
5. Durability modes
6. Async concurrency
7. Testing graphs without calling the API
8. Observability & deployment

In [1]:
from helpers import get_llm, show_graph, print_messages

llm = get_llm()   # ChatAnthropic -> Claude, key read from CLAUDE_KEY in .env

## 1. Retry policies

Network calls fail. Attach a `RetryPolicy` to a node: exponential backoff, only for the exceptions you choose.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy

calls = {"n": 0}

class S(TypedDict):
    result: str

def flaky_api(state: S):
    calls["n"] += 1
    print(f"attempt {calls['n']}")
    if calls["n"] < 3:
        raise ConnectionError("network blip")
    return {"result": "data received ✅"}

b = StateGraph(S)
b.add_node("flaky_api", flaky_api,
           retry_policy=RetryPolicy(max_attempts=5, initial_interval=0.2, backoff_factor=2.0, retry_on=ConnectionError))
b.add_edge(START, "flaky_api"); b.add_edge("flaky_api", END)
b.compile().invoke({})

attempt 1


attempt 2


attempt 3


{'result': 'data received ✅'}

## 2. Node caching

Skip re-running expensive, deterministic nodes when their input hasn't changed.

In [3]:
import time
from langgraph.types import CachePolicy
from langgraph.cache.memory import InMemoryCache

class Q(TypedDict):
    query: str
    answer: str

def expensive_lookup(state: Q):
    time.sleep(2)                                  # pretend this is slow
    return {"answer": f"result for {state['query']!r}"}

b = StateGraph(Q)
b.add_node("expensive_lookup", expensive_lookup, cache_policy=CachePolicy(ttl=120))
b.add_edge(START, "expensive_lookup"); b.add_edge("expensive_lookup", END)
cached_graph = b.compile(cache=InMemoryCache())

for q in ["langgraph", "langgraph", "python"]:
    t0 = time.time()
    cached_graph.invoke({"query": q})
    print(f"{q:<10} {time.time() - t0:.2f}s")

langgraph  2.03s
langgraph  0.00s


python     2.01s


## 3. Model fallbacks

If the primary model errors (outage, rate limit, bad config), fall back to another one.
Here the primary has a deliberately wrong model name.

In [4]:
from helpers import get_llm

broken = get_llm(model="claude-model-that-does-not-exist")
robust = broken.with_fallbacks([get_llm(effort="low")])
print(robust.invoke("Say 'fallback worked' and nothing else.").text)

fallback worked


## 4. Time travel: replay & fork

Every checkpoint can be revisited. **Replay** re-runs from a past checkpoint; **fork** edits the past state and continues on a new branch.
Great for debugging ("what if the agent had chosen differently?").

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

fast = get_llm(effort="low")

class JokeState(TypedDict):
    topic: str
    joke: str

def pick_topic(state: JokeState):
    return {"topic": fast.invoke("Suggest one random everyday topic for a joke. Reply with 1-3 words only.").text.strip()}

def write_joke(state: JokeState):
    return {"joke": fast.invoke(f"Write a one-line joke about {state['topic']}.").text}

b = StateGraph(JokeState)
b.add_node(pick_topic); b.add_node(write_joke)
b.add_edge(START, "pick_topic"); b.add_edge("pick_topic", "write_joke"); b.add_edge("write_joke", END)
tt_graph = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "time-travel"}}
print(tt_graph.invoke({"topic": ""}, cfg))

{'topic': 'Lost socks', 'joke': "I'm not saying my dryer eats socks, but it's never once complained about being hungry."}


In [6]:
history = list(tt_graph.get_state_history(cfg))
for s in history:
    print(f"step {s.metadata['step']:>2} | next={s.next!s:<18} | topic={s.values.get('topic')!r}")

before_joke = next(s for s in history if s.next == ("write_joke",))

step  2 | next=()                 | topic='Lost socks'
step  1 | next=('write_joke',)    | topic='Lost socks'
step  0 | next=('pick_topic',)    | topic=''
step -1 | next=('__start__',)     | topic=None


In [7]:
# 🔁 Replay: run again from the checkpoint before write_joke (same topic, new joke)
print("replay:", tt_graph.invoke(None, before_joke.config)["joke"])

# 🌿 Fork: change the past (topic) and continue on a new branch
fork_cfg = tt_graph.update_state(before_joke.config, {"topic": "penguins"})
print("fork  :", tt_graph.invoke(None, fork_cfg)["joke"])

replay: I'm not saying my dryer is a portal to another dimension, but somewhere out there is a sock kingdom, and it's ruled by my left argyle.


fork  : Penguins don't like talking to strangers at parties because they find it hard to break the ice.


## 5. Durability modes

How eagerly checkpoints are written:

| `durability=` | Writes checkpoints | Trade-off |
|---|---|---|
| `"exit"` | only when the run finishes/interrupts | fastest; can't resume from mid-run crashes |
| `"async"` (default) | in background while next step runs | good balance |
| `"sync"` | before each next step starts | safest; slower |

In [8]:
for mode in ["exit", "async", "sync"]:
    t0 = time.time()
    tt_graph.invoke({"topic": "durability"}, {"configurable": {"thread_id": f"dur-{mode}"}}, durability=mode)
    print(f"{mode:<6} {time.time() - t0:.2f}s")

exit   3.44s


async  3.70s


sync   3.27s


## 6. Async concurrency

Serving many users? Use `ainvoke` / `astream` and run requests concurrently.
`max_concurrency` in the config limits parallel tasks *inside* one run.

In [9]:
import asyncio

class QA(TypedDict):
    question: str
    answer: str

async def answer(state: QA):                       # nodes can be async functions
    reply = await fast.ainvoke(f"Answer in 3 words: {state['question']}")
    return {"answer": reply.text}

b = StateGraph(QA)
b.add_node(answer); b.add_edge(START, "answer"); b.add_edge("answer", END)
qa_graph = b.compile()

questions = ["Capital of France?", "Capital of Japan?", "Capital of Kenya?"]
t0 = time.time()
results = await asyncio.gather(*[qa_graph.ainvoke({"question": q}) for q in questions])
for r in results:
    print(f"{r['question']:<20} {r['answer']}")
print(f"\n{len(results)} runs concurrently in {time.time() - t0:.1f}s")

Capital of France?   The capital: Paris.
Capital of Japan?    It's Tokyo, Japan.
Capital of Kenya?    It's Nairobi, Kenya.

3 runs concurrently in 2.9s


## 7. Testing graphs without the API

Make the model an **argument** of your graph factory; in tests pass a fake model with scripted replies.
Fast, free, deterministic.

In [10]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langgraph.graph import MessagesState

def build_support_graph(model):
    def classify(state: MessagesState):
        return {"messages": [model.invoke(state["messages"])]}
    def route(state: MessagesState):
        return "escalate" if "URGENT" in state["messages"][-1].text else END
    def escalate(state: MessagesState):
        return {"messages": [AIMessage("Escalated to a human agent.")]}
    b = StateGraph(MessagesState)
    b.add_node(classify); b.add_node(escalate)
    b.add_edge(START, "classify")
    b.add_conditional_edges("classify", route, ["escalate", END])
    b.add_edge("escalate", END)
    return b.compile()

def test_urgent_is_escalated():
    fake = GenericFakeChatModel(messages=iter([AIMessage("URGENT: server down")]))
    out = build_support_graph(fake).invoke({"messages": [("user", "prod is down!")]})
    assert out["messages"][-1].text == "Escalated to a human agent."

def test_normal_is_not_escalated():
    fake = GenericFakeChatModel(messages=iter([AIMessage("normal question")]))
    out = build_support_graph(fake).invoke({"messages": [("user", "how do I reset my password?")]})
    assert out["messages"][-1].text == "normal question"

test_urgent_is_escalated(); test_normal_is_not_escalated()
print("✅ all tests passed")

✅ all tests passed


## 8. Observability & deployment

**Tracing with LangSmith** — add to `.env` and every node, LLM call and tool call is traced:
```
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=langgraph-practical
```

**Serving a graph as an API** — create `langgraph.json`:
```json
{
  "dependencies": ["."],
  "graphs": {"support_agent": "./agent.py:graph"},
  "env": ".env"
}
```
then `pip install "langgraph-cli[inmem]"` and `langgraph dev` → a local server with streaming, threads,
persistence and a visual Studio UI. Use `PostgresSaver` / `PostgresStore` in production.

## ✅ Production checklist
- [ ] Retry policies on network-bound nodes, fallbacks on models
- [ ] Durable checkpointer (Postgres) + appropriate `durability`
- [ ] `recursion_limit` and step limits on agent loops
- [ ] Human approval on irreversible actions
- [ ] Tests with fake models; tracing in staging/prod